#Silver Wyckoff Features

In [0]:
    %run ../config/config

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType,
    DoubleType, BooleanType, IntegerType,
)

#### 1. Read inputs

In [0]:
input_cols_cleaned = ["symbol", "date", "open", "high", "low", "close", "volume"]
input_cols_indicators = ["symbol", "date", "sma_20", "sma_50", "atr_14"]

cleaned_df = spark.table(tables["cleaned_ohlcv"]).select(*input_cols_cleaned)
indicators_df = spark.table(tables["technical_indicators"]).select(*input_cols_indicators)

# cleaned OHLCV + technical indicators 
joined_df = cleaned_df.join(indicators_df, on=["symbol", "date"], how="inner")

#### 2. Wyckoff feature 

In [0]:
wyckoff_schema = StructType([
    StructField("symbol", StringType(), False),
    StructField("date", DateType(), False),
    StructField("price_spread", DoubleType()),
    StructField("close_position", DoubleType()),
    StructField("volume_climax_up", BooleanType()),
    StructField("volume_climax_down", BooleanType()),
    StructField("stopping_volume", BooleanType()),
    StructField("no_demand", BooleanType()),
    StructField("no_supply", BooleanType()),
    StructField("spring_signal", BooleanType()),
    StructField("upthrust_signal", BooleanType()),
    StructField("sos_signal", BooleanType()),
    StructField("sow_signal", BooleanType()),
    StructField("effort_vs_result", DoubleType()),
    StructField("test_signal", BooleanType()),
    StructField("support_level", DoubleType()),
    StructField("resistance_level", DoubleType()),
    StructField("trading_range_width", DoubleType()),
    StructField("trend_direction", IntegerType()),
    StructField("days_in_range", DoubleType()),
    StructField("volume_trend", DoubleType()),
    StructField("relative_position", DoubleType()),
    StructField("breakout_strength", DoubleType()),
    StructField("volume_price_divergence", DoubleType()),
])


def compute_wyckoff_features(pdf: pd.DataFrame) -> pd.DataFrame:
    """Compute Wyckoff-specific features for one symbol."""
    pdf = pdf.sort_values("date").reset_index(drop=True)
    n = len(pdf)

    o = pdf["open"].values
    h = pdf["high"].values
    l = pdf["low"].values
    c = pdf["close"].values
    v = pdf["volume"].astype(float).values
    sma_20 = pdf["sma_20"].values if "sma_20" in pdf.columns else np.full(n, np.nan)
    sma_50 = pdf["sma_50"].values if "sma_50" in pdf.columns else np.full(n, np.nan)

    # price spread 
    spread = h - l
    price_spread = np.where(c > 0, spread / c, 0.0)  

    # close position within bar: (close - low) / (high - low)
    bar_range = h - l
    close_position = np.where(bar_range > 0, (c - l) / bar_range, 0.5)

    # volume metrics 
    vol_sma_20 = pd.Series(v).rolling(20, min_periods=1).mean().values
    vol_ratio = np.where(vol_sma_20 > 0, v / vol_sma_20, 1.0)

    spread_sma_20 = pd.Series(spread).rolling(20, min_periods=1).mean().values
    spread_ratio = np.where(spread_sma_20 > 0, spread / spread_sma_20, 1.0)

    # up bar: close > open
    # down bar: close < open
    is_up = c > o
    is_down = c < o

    # volume climax up: high volume + wide spread + up bar 
    volume_climax_up = (vol_ratio > 2.0) & (spread_ratio > 1.5) & is_up

    # volume climax down: high volume + wide spread + down bar
    volume_climax_down = (vol_ratio > 2.0) & (spread_ratio > 1.5) & is_down

    # stopping volume: high volume + narrow spread 
    stopping_volume = (vol_ratio > 1.5) & (spread_ratio < 0.7)

    # no demand: low volume + narrow spread + up bar
    no_demand = (vol_ratio < 0.7) & (spread_ratio < 0.7) & is_up

    # no supply: low volume + narrow spread + down bar 
    no_supply = (vol_ratio < 0.7) & (spread_ratio < 0.7) & is_down

    # Support & Resistance 
    support_level = pd.Series(l).rolling(60, min_periods=20).min().shift(1).values
    resistance_level = pd.Series(h).rolling(60, min_periods=20).max().shift(1).values
    trading_range_width = np.where(
        (support_level > 0) & ~np.isnan(support_level),
        (resistance_level - support_level) / support_level,0.0,)

    # Spring signal
    spring_signal = (
        ~np.isnan(support_level) &
        (l < support_level) &
        (c > support_level) &
        (close_position > 0.4) &
        (vol_ratio < 1.0)
    )
    spring_signal[:20] = False

    # Upthrust Signal
    upthrust_signal = (
        ~np.isnan(resistance_level) &
        (h > resistance_level) &
        (c < resistance_level) &
        (close_position < 0.6) &
        (vol_ratio < 1.0)
    )
    upthrust_signal[:20] = False

    # Sign of Strength (SOS)
    sos_signal = (
        ~np.isnan(resistance_level) &
        (c > resistance_level) &
        is_up &
        (vol_ratio > 1.2)
    )
    sos_signal[:20] = False

    # Sign of Weakness (SOW)
    sow_signal = (
        ~np.isnan(support_level) &
        (c < support_level) &
        is_down &
        (vol_ratio > 1.2)
    )
    sow_signal[:20] = False

    # Effort vs result: volume change vs price change ratio
    price_change = np.abs(np.diff(c, prepend=c[0]))
    vol_change = np.abs(np.diff(v, prepend=v[0]))
    pc_norm = np.where(c > 0, price_change / c, 0)
    vc_norm = np.where(vol_sma_20 > 0, vol_change / vol_sma_20, 0)
    effort_vs_result = np.where(pc_norm > 1e-9, vc_norm / pc_norm, 0.0)
    effort_vs_result = np.clip(effort_vs_result, 0, 10)

    # Test Signal: revisit support/resistance on low volume
    safe_support = np.where(
        (support_level > 0) & ~np.isnan(support_level), support_level, 1.0
    )
    safe_resistance = np.where(
        (resistance_level > 0) & ~np.isnan(resistance_level), resistance_level, 1.0
    )
    near_support = np.abs(l - support_level) / safe_support < 0.02
    near_resistance = np.abs(h - resistance_level) / safe_resistance < 0.02
    test_signal = (
        ~np.isnan(support_level) &
        (near_support | (near_resistance & ~np.isnan(resistance_level))) &
        (vol_ratio < 0.7)
    )
    test_signal[:20] = False

    # SMA20 vs SMA50
    trend_direction = np.where(
        np.isnan(sma_20) | np.isnan(sma_50), 0,
        np.where(sma_20 > sma_50 * 1.005, 1,
                 np.where(sma_20 < sma_50 * 0.995, -1, 0))
    ).astype(int)

    # Phase C/D distinguishing features 
    in_range = (
        ~np.isnan(support_level) & ~np.isnan(resistance_level) &
        (l >= support_level * 0.98) & (h <= resistance_level * 1.02)
    )

    # Vectorized consecutive-day counter
    _ir = pd.Series(in_range.astype(int))
    days_in_range = _ir.groupby((~in_range).cumsum()).cumsum().values.astype(float)

    # Volume Trend: slope of volume SMA over 20 bars (rising/falling)
    vol_sma_series = pd.Series(vol_sma_20)
    vol_sma_diff = vol_sma_series.diff(10).values  # 10-bar momentum
    volume_trend = np.where(
        vol_sma_20 > 0, vol_sma_diff / vol_sma_20, 0.0
    )
    volume_trend = np.nan_to_num(volume_trend, nan=0.0)

    # Relative Position: where close sits within the trading range [0, 1]
    range_width = resistance_level - support_level
    relative_position = np.where(
        (~np.isnan(range_width)) & (range_width > 0),
        (c - support_level) / range_width,
        0.5, 
    )
    relative_position = np.clip(relative_position, -0.5, 1.5)

    # Breakout Strength: (close - resistance) / ATR 
    # Phase D accumulation = positive breakout, phase D distribution = negative
    atr_14 = pdf["atr_14"].values if "atr_14" in pdf.columns else np.ones(n)
    safe_atr = np.where((atr_14 > 0) & ~np.isnan(atr_14), atr_14, 1.0)
    breakout_strength = np.where(
        ~np.isnan(resistance_level), (c - resistance_level) / safe_atr, 0.0,)
    breakout_strength = np.clip(breakout_strength, -5, 5)

    # Volume-Price Divergence: price direction vs volume direction
    price_ret_5 = pd.Series(c).pct_change(5).values
    vol_ret_5 = pd.Series(v).pct_change(5).values
    volume_price_divergence = np.where(
        (np.abs(price_ret_5) > 1e-6) & ~np.isnan(price_ret_5) & ~np.isnan(vol_ret_5),
        np.sign(price_ret_5) * vol_ret_5,  # positive = aligned, negative = divergent
        0.0,
    )
    volume_price_divergence = np.clip(volume_price_divergence, -5, 5)

    # Build result
    result = pd.DataFrame({
        "symbol": pdf["symbol"],
        "date": pdf["date"],
        "price_spread": price_spread,
        "close_position": close_position,
        "volume_climax_up": volume_climax_up,
        "volume_climax_down": volume_climax_down,
        "stopping_volume": stopping_volume,
        "no_demand": no_demand,
        "no_supply": no_supply,
        "spring_signal": spring_signal,
        "upthrust_signal": upthrust_signal,
        "sos_signal": sos_signal,
        "sow_signal": sow_signal,
        "effort_vs_result": effort_vs_result,
        "test_signal": test_signal,
        "support_level": support_level,
        "resistance_level": resistance_level,
        "trading_range_width": trading_range_width,
        "trend_direction": trend_direction,
        "days_in_range": days_in_range,
        "volume_trend": volume_trend,
        "relative_position": relative_position,
        "breakout_strength": breakout_strength,
        "volume_price_divergence": volume_price_divergence,
    })

    return result

#### 3. Apply to All Symbols

In [0]:
wyckoff_df = joined_df.groupBy("symbol").applyInPandas(compute_wyckoff_features, schema=wyckoff_schema)

#### 4. Write to Silver

In [0]:
wyckoff_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tables["wyckoff_features"])
print(f"Successfull written to {tables['wyckoff_features']}")

#### 5. Verify

In [0]:
%sql
SELECT 
    COUNT(*) AS total_rows, 
    COUNT(DISTINCT symbol) AS unique_symbols
FROM wyckoff_catalog.silver.wyckoff_features;

In [0]:
%sql
SELECT
    COUNT(*)                               AS total,
    SUM(CAST(volume_climax_up AS INT))     AS climax_up,
    SUM(CAST(volume_climax_down AS INT))   AS climax_down,
    SUM(CAST(stopping_volume AS INT))      AS stopping_vol,
    SUM(CAST(no_demand AS INT))            AS no_demand,
    SUM(CAST(no_supply AS INT))            AS no_supply,
    SUM(CAST(spring_signal AS INT))        AS spring,
    SUM(CAST(upthrust_signal AS INT))      AS upthrust,
    SUM(CAST(sos_signal AS INT))           AS sos,
    SUM(CAST(sow_signal AS INT))           AS sow,
    SUM(CAST(test_signal AS INT))          AS test
FROM wyckoff_catalog.silver.wyckoff_features;

In [0]:
%sql
SELECT *
FROM wyckoff_catalog.silver.wyckoff_features
WHERE symbol = 'HPG'
ORDER BY date DESC
LIMIT 20;